# Dermal Absorption of Caffeine
**Skin Layers · Hair Follicle Pathway · Transdermal PK | OSP MoBi Exercise**

**Author:** Nadia Tasnim Ahmed, PhD  
**Field:** PBPK · Dermal Absorption · Transdermal Drug Delivery  
**Tools:** Python · numpy · scipy · pandas · matplotlib · plotly  
**Reference:** OSP MoBi Course — Dermal Absorption of Caffeine (v11)

---

## Background

Dermal absorption occurs through two parallel pathways:

```
Topical dose (skin surface)
    │
    ├── Transcellular pathway (intercellular lipid route):
    │   Stratum corneum → Viable epidermis → Dermis → Capillaries → Blood
    │
    └── Hair follicle shunt pathway:
        Hair follicle → Follicular infundibulum → Dermis → Blood
        (faster route — bypasses stratum corneum barrier)
```

**Skin layer structure (MoBi containers):**

| Layer | Thickness | Role |
|---|---|---|
| Stratum corneum (SC) | 10-20 μm | Main barrier (dead cells, lipid bilayers) |
| Viable epidermis (VE) | 50-100 μm | Metabolically active, no blood supply |
| Dermis | 1-4 mm | Blood vessels, lymphatics |
| Subcutaneous fat | 5-30 mm | Buffer reservoir |

**Caffeine properties:**
- MW=194.19, logP=-0.07 (hydrophilic) → poor passive penetration
- Hair follicle pathway is DOMINANT for caffeine
- Widely used as a model compound in dermal absorption studies
- Applied topically on chest in this exercise (hair-bearing skin)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import odeint
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded.')

## 1. Caffeine Drug Properties & Skin Parameters

In [ ]:
# Caffeine physicochemical properties
CAFFEINE = dict(
    name     = 'Caffeine',
    MW       = 194.19,
    logP     = -0.07,     # hydrophilic
    pKa      = 14.0,      # neutral
    fu       = 0.65,      # 35% plasma protein bound
    B2P      = 0.92,
    # Systemic PK
    Vc_kg    = 0.60,      # L/kg
    Vp_kg    = 0.10,
    Q_sys_kg = 0.05,
    CL_kg    = 0.078,     # L/h/kg hepatic (CYP1A2)
    # Dermal application
    dose_dermal = 2.0,    # mg/cm2 applied concentration
    area_chest  = 500.0,  # cm2 chest area (hair-bearing)
)
BW = 70.0
DOSE_DERMAL = CAFFEINE['dose_dermal'] * CAFFEINE['area_chest']  # mg

# ── Skin compartment parameters (chest, hair-bearing) ─────────────────
SKIN = dict(
    # Layer thicknesses (cm)
    L_SC   = 0.0015,   # stratum corneum 15 μm
    L_VE   = 0.008,    # viable epidermis 80 μm
    L_dermis= 0.20,    # dermis 2 mm
    L_subcut= 1.50,    # subcutaneous 15 mm
    # Volumes (cm2 * thickness * density)
    area   = CAFFEINE['area_chest'],  # cm2
    # Diffusion coefficients in each layer (cm2/h)
    D_SC   = 1.5e-8,   # very slow in SC (lipid barrier)
    D_VE   = 5.0e-6,
    D_dermis=8.0e-5,
    # Partition coefficients (layer:vehicle or layer:water)
    Ksc_w  = 0.05,     # SC:water — hydrophilic stays in water
    Kve_w  = 1.20,
    Kd_w   = 1.50,
    # Permeability-surface area products (cm3/h = mL/h)
    PS_SC_VE   = 0.008,   # SC → VE (rate-limiting)
    PS_VE_derm = 1.20,    # VE → dermis
    PS_derm_bld= 8.50,    # dermis → blood capillaries
    # Hair follicle parameters
    N_follicles  = 80.0,      # follicles/cm2 on chest
    r_follicle   = 0.004,     # cm radius of follicle
    L_follicle   = 0.35,      # cm depth
    PS_follicle  = 0.25,      # mL/h per follicle (shunt route)
)

# Volumes of each skin layer (mL)
area = SKIN['area']
SKIN['V_SC']    = area * SKIN['L_SC']    * 1.0  # density ~1
SKIN['V_VE']    = area * SKIN['L_VE']    * 1.0
SKIN['V_dermis']= area * SKIN['L_dermis']* 1.0
SKIN['V_subcut']= area * SKIN['L_subcut']* 0.95
# Total hair follicle volume
SKIN['V_follicle'] = (SKIN['N_follicles'] * area *
                      np.pi * SKIN['r_follicle']**2 * SKIN['L_follicle'])
SKIN['PS_foll_total'] = SKIN['PS_follicle'] * SKIN['N_follicles'] * area

print('Caffeine dermal model parameters:')
print('  Dose applied:      ', DOSE_DERMAL, 'mg')
print('  Application area:  ', area, 'cm2 (chest)')
print(f'  V_SC:              {SKIN["V_SC"]:.4f} mL')
print(f'  V_VE:              {SKIN["V_VE"]:.4f} mL')
print(f'  V_dermis:          {SKIN["V_dermis"]:.3f} mL')
print(f'  Hair follicles:    {int(SKIN["N_follicles"]*area)} total on chest')
print(f'  V_follicle:        {SKIN["V_follicle"]:.4f} mL')
print(f'  PS_follicle total: {SKIN["PS_foll_total"]:.3f} mL/h')

## 2. Dermal PBPK ODE System

**MoBi containers:** Surface → SC → VE → Dermis → Blood  
**MoBi shunt:**      Surface → Hair follicle → Dermis → Blood

In [ ]:
def dermal_pbpk_odes(y, t, p):
    """
    Dermal absorption PBPK for caffeine.

    State variables (amounts in mg):
    y[0] = A_surface   : dose on skin surface
    y[1] = A_SC        : stratum corneum
    y[2] = A_VE        : viable epidermis
    y[3] = A_follicle  : hair follicle shunt
    y[4] = A_dermis    : dermis
    y[5] = Ac_sys      : systemic central
    y[6] = Ap_sys      : systemic peripheral
    """
    A_surf, A_SC, A_VE, A_foll, A_derm, Ac, Ap = y

    # Concentrations (mg/mL)
    C_surf = max(A_surf / p['V_surf'], 0)
    C_SC   = max(A_SC   / p['V_SC'],   0)
    C_VE   = max(A_VE   / p['V_VE'],   0)
    C_foll = max(A_foll / p['V_foll'], 0)
    C_derm = max(A_derm / p['V_derm'], 0)
    Cc_sys = max(Ac     / p['Vc_sys'], 0)
    Cp_sys = max(Ap     / p['Vp_sys'], 0)

    # ── Transcellular pathway (SC is rate-limiting) ───────────────────
    J_surf_SC = p['PS_SC']   * (C_surf - C_SC  / p['Ksc'])
    J_SC_VE   = p['PS_VE']   * (C_SC   / p['Ksc'] - C_VE)
    J_VE_derm = p['PS_derm'] * (C_VE   - C_derm)

    # ── Hair follicle shunt pathway ───────────────────────────────────
    J_surf_foll = p['PS_foll'] * (C_surf - C_foll)   # surface → follicle
    J_foll_derm = p['PS_foll'] * (C_foll - C_derm)   # follicle → dermis

    # ── Dermis → blood ────────────────────────────────────────────────
    J_derm_bld  = p['PS_cap']  * (C_derm - Cc_sys / p['Kp_derm'])

    # ── Systemic distribution & elimination ───────────────────────────
    Q_sys = p['Q_sys']
    CL_t  = p['CL_sys']

    dA_surf = -J_surf_SC - J_surf_foll
    dA_SC   =  J_surf_SC - J_SC_VE
    dA_VE   =  J_SC_VE   - J_VE_derm
    dA_foll =  J_surf_foll - J_foll_derm
    dA_derm =  J_VE_derm + J_foll_derm - J_derm_bld
    dAc_sys =  J_derm_bld - CL_t*Cc_sys - Q_sys*(Cc_sys - Cp_sys/p['Kp_sys'])
    dAp_sys =  Q_sys*(Cc_sys - Cp_sys/p['Kp_sys'])

    return [dA_surf, dA_SC, dA_VE, dA_foll, dA_derm, dAc_sys, dAp_sys]


# Build parameters
V_surf = DOSE_DERMAL / (CAFFEINE['dose_dermal'] * 1.0)  # mL (treat as thin film)
p_base = dict(
    V_surf=max(V_surf, 0.5),
    V_SC=SKIN['V_SC'], V_VE=SKIN['V_VE'],
    V_foll=SKIN['V_follicle'],
    V_derm=SKIN['V_dermis'],
    Vc_sys=CAFFEINE['Vc_kg']*BW*1000,  # mL
    Vp_sys=CAFFEINE['Vp_kg']*BW*1000,
    PS_SC=SKIN['PS_SC_VE'],
    PS_VE=SKIN['PS_VE_derm'],
    PS_derm=SKIN['PS_VE_derm'],
    PS_foll=SKIN['PS_foll_total'],
    PS_cap=SKIN['PS_derm_bld'],
    Ksc=SKIN['Ksc_w'], Kp_derm=1.5, Kp_sys=1.2,
    CL_sys=CAFFEINE['CL_kg']*BW*1000,  # mL/h
    Q_sys=CAFFEINE['Q_sys_kg']*BW*1000,
)

t_sim  = np.linspace(0, 24, 2000)
y0     = [DOSE_DERMAL, 0, 0, 0, 0, 0, 0]

# With hair follicles (normal chest skin)
sol_hair = odeint(dermal_pbpk_odes, y0, t_sim, args=(p_base,),
                  rtol=1e-8, atol=1e-10)

# Without hair follicles (hairless / shaved / occluded)
p_nohair = p_base.copy()
p_nohair['PS_foll'] = 0.0
sol_nohair = odeint(dermal_pbpk_odes, y0, t_sim, args=(p_nohair,),
                    rtol=1e-8, atol=1e-10)

# Extract systemic concentrations (mg/L = ug/mL)
C_sys_hair   = np.maximum(sol_hair[:,5]   / (p_base['Vc_sys']/1000), 0)
C_sys_nohair = np.maximum(sol_nohair[:,5] / (p_base['Vc_sys']/1000), 0)

# Fractional absorption
F_hair   = 1 - sol_hair[-1,0]   / DOSE_DERMAL
F_nohair = 1 - sol_nohair[-1,0] / DOSE_DERMAL

AUC_hair   = np.trapezoid(C_sys_hair,   t_sim)
AUC_nohair = np.trapezoid(C_sys_nohair, t_sim)

print('Dermal absorption results (2 mg/cm2 on 500 cm2 chest, 24h):')
print('  With hair follicles:')
print('    Cmax:', round(C_sys_hair.max()*1000,4), 'ug/L')
print('    AUC: ', round(AUC_hair,4), 'mg*h/L')
print('    F_abs:', round(F_hair*100,2), '%')
print('  Without hair (transcellular only):')
print('    Cmax:', round(C_sys_nohair.max()*1000,4), 'ug/L')
print('    AUC: ', round(AUC_nohair,4), 'mg*h/L')
print('    F_abs:', round(F_nohair*100,2), '%')
print('  Hair follicle contribution to absorption:')
print('    AUC ratio (hair/no-hair):', round(AUC_hair/max(AUC_nohair,1e-8),2))

## 3. Pathway Analysis — Follicle vs Transcellular

In [ ]:
# Flux through each pathway
C_surf_t = np.maximum(sol_hair[:,0] / p_base['V_surf'], 0)
C_SC_t   = np.maximum(sol_hair[:,1] / SKIN['V_SC'],     0)
C_VE_t   = np.maximum(sol_hair[:,2] / SKIN['V_VE'],     0)
C_foll_t = np.maximum(sol_hair[:,3] / SKIN['V_follicle'],0)
C_derm_t = np.maximum(sol_hair[:,4] / SKIN['V_dermis'], 0)

# Pathway fluxes (mg/h)
J_transcellular = p_base['PS_SC']   * (C_surf_t - C_SC_t / SKIN['Ksc_w'])
J_follicle      = p_base['PS_foll'] * (C_surf_t - C_foll_t)
J_total         = J_transcellular + J_follicle

# Cumulative absorption
cum_trans = np.cumsum(np.maximum(J_transcellular,0)) * (t_sim[1]-t_sim[0])
cum_foll  = np.cumsum(np.maximum(J_follicle,0))      * (t_sim[1]-t_sim[0])

print('Pathway flux analysis:')
print('  Peak transcellular flux:', round(max(J_transcellular)*1000,4), 'ug/h')
print('  Peak follicle flux:     ', round(max(J_follicle)*1000,4),      'ug/h')
print('  Follicle/transcellular at t=1h:',
      round(J_follicle[50]/max(J_transcellular[50],1e-10),1), 'x faster')

# Follicle density sensitivity
densities = [0, 20, 40, 80, 120, 200]  # follicles/cm2
auc_by_density = []
for density in densities:
    p_d = p_base.copy()
    p_d['PS_foll'] = SKIN['PS_follicle'] * density * area
    sol_d = odeint(dermal_pbpk_odes, y0, t_sim, args=(p_d,),
                   rtol=1e-6, atol=1e-8)
    C_d   = np.maximum(sol_d[:,5] / (p_base['Vc_sys']/1000), 0)
    auc_by_density.append(np.trapezoid(C_d, t_sim))

print('\nFollicle density sensitivity (AUC mg*h/L):')
for d, auc in zip(densities, auc_by_density):
    bar = '#' * int(auc/max(auc_by_density)*30)
    print(f'  {str(d).rjust(4)} foll/cm2: {round(auc,6):.6f}  {bar}')

## 4. Occlusion Effect & Multiple Applications

In [ ]:
# Occlusion increases SC hydration → faster penetration
p_occluded = p_base.copy()
p_occluded['PS_SC'] *= 5.0   # occlusion increases SC permeability ~5x
p_occluded['PS_foll'] *= 1.5  # follicle slightly enhanced

sol_occ = odeint(dermal_pbpk_odes, y0, t_sim, args=(p_occluded,),
                 rtol=1e-8, atol=1e-10)
C_occ   = np.maximum(sol_occ[:,5] / (p_base['Vc_sys']/1000), 0)
AUC_occ = np.trapezoid(C_occ, t_sim)

print('Occlusion effect:')
print('  Open:    AUC =', round(AUC_hair,6), 'mg*h/L')
print('  Occluded:AUC =', round(AUC_occ,6),  'mg*h/L')
print('  Ratio:        ', round(AUC_occ/max(AUC_hair,1e-8),2), 'x increase')

# Skin region comparison (follicle density varies by body site)
BODY_SITES = {
    'Scalp':    dict(density=300, area=600),
    'Chest':    dict(density=80,  area=500),
    'Forearm':  dict(density=20,  area=300),
    'Palm':     dict(density=0,   area=100),  # hairless
}

print('\nBody site comparison (same dose/area):')
print('Site'.ljust(10), 'Density(/cm2)', 'AUC(mg*h/L)', 'F_abs(%)')
for site, props in BODY_SITES.items():
    p_site = p_base.copy()
    p_site['V_surf']  = props['area'] * 2.0
    p_site['PS_foll'] = SKIN['PS_follicle'] * props['density'] * props['area']
    y0_site = [CAFFEINE['dose_dermal'] * props['area']] + [0]*6
    sol_site= odeint(dermal_pbpk_odes, y0_site, t_sim, args=(p_site,),
                     rtol=1e-6, atol=1e-8)
    C_site  = np.maximum(sol_site[:,5] / (p_base['Vc_sys']/1000), 0)
    auc_s   = np.trapezoid(C_site, t_sim)
    f_s     = (1 - sol_site[-1,0]/(CAFFEINE['dose_dermal']*props['area']))*100
    print(site.ljust(10),
          str(props['density']).rjust(13),
          str(round(auc_s,7)).rjust(12),
          str(round(f_s,3)).rjust(9)+'%')

## 5. Visualization

In [ ]:
BLUE='#2563EB'; RED='#DC2626'; GREEN='#16A34A'
AMBER='#D97706'; PURP='#7C3AED'; TEAL='#0D9488'

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, hspace=0.45, wspace=0.38)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])
ax4 = fig.add_subplot(gs[1, 2])
ax5 = fig.add_subplot(gs[2, 0])
ax6 = fig.add_subplot(gs[2, 1])
ax7 = fig.add_subplot(gs[2, 2])

# Panel 1: Systemic PK — with vs without hair
ax1.plot(t_sim, C_sys_hair*1e3,   color=RED,  lw=2.5,
         label='With hair follicles (chest)')
ax1.plot(t_sim, C_sys_nohair*1e3, color=BLUE, lw=2.5, ls='--',
         label='Without hair (transcellular only)')
ax1.plot(t_sim, C_occ*1e3,        color=GREEN, lw=2, ls='-.',
         label='Occluded + hair')
ax1.set(xlabel='Time (h)', ylabel='Caffeine plasma conc (ug/L)',
        title='Systemic Caffeine PK After Dermal Application\n'
              '2 mg/cm2 on 500 cm2 chest | With vs Without Hair Follicles')
ax1.title.set_fontweight('bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.25)

# Panel 2: Skin layer concentrations
ax2.semilogy(t_sim, C_surf_t+1e-10, color='black', lw=2,  label='Surface')
ax2.semilogy(t_sim, C_SC_t+1e-10,   color=AMBER, lw=2,   label='SC')
ax2.semilogy(t_sim, C_VE_t+1e-10,   color=GREEN, lw=2,   label='VE')
ax2.semilogy(t_sim, C_foll_t+1e-10, color=PURP,  lw=2,   label='Follicle')
ax2.semilogy(t_sim, C_derm_t+1e-10, color=RED,   lw=2,   label='Dermis')
ax2.semilogy(t_sim, C_sys_hair+1e-10,color=BLUE, lw=2,   label='Plasma')
ax2.set(xlabel='Time (h)', ylabel='Concentration (mg/mL) [log]',
        title='Skin Layer Concentrations\n(Semi-log, with hair)')
ax2.title.set_fontweight('bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.25, which='both')

# Panel 3: Pathway fluxes
ax3.fill_between(t_sim, J_transcellular*1000, alpha=0.4,
                  color=BLUE, label='Transcellular flux')
ax3.fill_between(t_sim, J_follicle*1000, alpha=0.4,
                  color=RED, label='Follicle shunt flux')
ax3.plot(t_sim, J_transcellular*1000, color=BLUE, lw=2)
ax3.plot(t_sim, J_follicle*1000,      color=RED,  lw=2)
ax3.set(xlabel='Time (h)', ylabel='Flux (ug/h)',
        title='Absorption Pathway Fluxes\nTranscellular vs Follicle Shunt')
ax3.title.set_fontweight('bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.25)

# Panel 4: Cumulative absorption
ax4.plot(t_sim, cum_trans*1000, color=BLUE, lw=2, label='Transcellular')
ax4.plot(t_sim, cum_foll*1000,  color=RED,  lw=2, label='Follicle shunt')
ax4.plot(t_sim, (cum_trans+cum_foll)*1000, color='black',
         lw=2.5, ls='--', label='Total absorbed')
ax4.set(xlabel='Time (h)', ylabel='Cumulative absorption (ug)',
        title='Cumulative Absorption\nby Pathway')
ax4.title.set_fontweight('bold')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.25)

# Panel 5: Follicle density sensitivity
ax5.plot(densities, [a*1e6 for a in auc_by_density],
         'o-', color=PURP, lw=2.5, ms=10)
ax5.axvline(80, color='gray', ls='--', lw=1.5, label='Chest (80/cm2)')
ax5.set(xlabel='Hair follicle density (/cm2)',
        ylabel='Systemic AUC (× 10⁻⁶ mg·h/L)',
        title='Follicle Density Sensitivity\n(AUC vs follicle density)')
ax5.title.set_fontweight('bold')
ax5.legend(fontsize=9); ax5.grid(True, alpha=0.25)

# Panel 6: Body site comparison
site_names    = list(BODY_SITES.keys())
site_densities= [BODY_SITES[s]['density'] for s in site_names]
site_colors   = [RED, AMBER, GREEN, BLUE]
ax6.bar(site_names, site_densities, color=site_colors, alpha=0.85)
ax6.set(ylabel='Hair follicle density (/cm2)',
        title='Body Site Comparison\n(Follicle density by region)')
ax6.title.set_fontweight('bold'); ax6.grid(True, alpha=0.25, axis='y')

# Panel 7: Model diagram
ax7.axis('off')
diagram = (
    'DERMAL ABSORPTION MODEL\n\n'
    'Surface (dose applied)\n'
    '  ↓ PS_SC (transcellular)    ↓ PS_foll (shunt)\n'
    'Stratum corneum              Hair follicle\n'
    '  ↓ (slow, rate-limiting)     ↓ (fast, bypasses SC)\n'
    'Viable epidermis              ↓\n'
    '  ↓                          Dermis\n'
    'Dermis ─────────────────────► Blood\n\n'
    f'Caffeine: logP={CAFFEINE["logP"]}\n'
    '  → hydrophilic → poor SC penetration\n'
    '  → follicle shunt DOMINATES\n\n'
    f'Chest: {int(SKIN["N_follicles"]*area)} follicles\n'
    f'  PS_SC:   {SKIN["PS_SC_VE"]:.3f} mL/h\n'
    f'  PS_foll: {SKIN["PS_foll_total"]:.2f} mL/h'
)
ax7.text(0.05, 0.95, diagram, transform=ax7.transAxes,
         fontsize=9, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax7.set_title('Model Structure', fontweight='bold')

plt.suptitle(
    'Dermal Absorption of Caffeine — Skin Layer Model & Hair Follicle Shunt\n'
    'Transcellular vs Follicle Pathway · Occlusion · Body Site | OSP MoBi Exercise',
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig('dermal_caffeine_pbpk.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dermal_caffeine_pbpk.png')

## 6. Interactive Dashboard

In [ ]:
fig_p = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Systemic PK: With vs Without Hair',
        'Skin Layer Concentrations',
        'Absorption Pathway Fluxes',
        'Follicle Density Sensitivity'
    ),
    vertical_spacing=0.18, horizontal_spacing=0.12
)

for arr, name, color, dash in [
    (C_sys_hair*1e3,   'With hair',     RED,   'solid'),
    (C_sys_nohair*1e3, 'Without hair',  BLUE,  'dash'),
    (C_occ*1e3,        'Occluded+hair', GREEN, 'dot'),
]:
    fig_p.add_trace(go.Scatter(
        x=t_sim, y=arr, mode='lines', name=name,
        line=dict(color=color, width=2, dash=dash),
        hovertemplate=name+': %{x:.1f}h=%{y:.5f} ug/L<extra></extra>'
    ), row=1, col=1)

for arr, name, color in [
    (C_surf_t, 'Surface', 'black'),
    (C_SC_t,   'SC',      AMBER),
    (C_VE_t,   'VE',      GREEN),
    (C_foll_t, 'Follicle',PURP),
    (C_derm_t, 'Dermis',  RED),
    (C_sys_hair,'Plasma', BLUE),
]:
    fig_p.add_trace(go.Scatter(
        x=t_sim, y=arr+1e-10, mode='lines', name=name,
        line=dict(color=color, width=2),
        hovertemplate=name+': %{x:.1f}h=%{y:.8f}<extra></extra>'
    ), row=1, col=2)

fig_p.add_trace(go.Scatter(
    x=t_sim, y=J_transcellular*1000, mode='lines',
    name='Transcellular', fill='tozeroy',
    line=dict(color=BLUE, width=2), fillcolor='rgba(37,99,235,0.2)'
), row=2, col=1)
fig_p.add_trace(go.Scatter(
    x=t_sim, y=J_follicle*1000, mode='lines',
    name='Follicle shunt', fill='tozeroy',
    line=dict(color=RED, width=2), fillcolor='rgba(220,38,38,0.2)'
), row=2, col=1)

fig_p.add_trace(go.Scatter(
    x=densities, y=[a*1e6 for a in auc_by_density],
    mode='lines+markers', name='AUC vs density',
    line=dict(color=PURP, width=2), marker=dict(size=10)
), row=2, col=2)

for ri,ci,xl,yl in [
    (1,1,'Time (h)','Conc (ug/L)'),
    (1,2,'Time (h)','Conc (mg/mL)'),
    (2,1,'Time (h)','Flux (ug/h)'),
    (2,2,'Follicles/cm2','AUC (×10⁻⁶)')
]:
    fig_p.update_xaxes(title_text=xl, row=ri, col=ci)
    fig_p.update_yaxes(title_text=yl, row=ri, col=ci)
fig_p.update_yaxes(type='log', row=1, col=2)

fig_p.update_layout(
    title=dict(
        text='Dermal Caffeine Absorption — Interactive Dashboard<br>'
             '<sup>Transcellular vs follicle shunt | Skin layers | Body site | OSP MoBi Exercise</sup>',
        font=dict(size=14)
    ),
    height=720, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, x=0)
)
fig_p.show()
fig_p.write_html('dermal_caffeine_dashboard.html')
print('Saved: dermal_caffeine_dashboard.html')

## 7. Export

In [ ]:
df_out = pd.DataFrame({
    'time_h':          t_sim,
    'C_plasma_hair':   C_sys_hair*1e3,
    'C_plasma_nohair': C_sys_nohair*1e3,
    'C_plasma_occluded':C_occ*1e3,
    'C_SC':            C_SC_t,
    'C_VE':            C_VE_t,
    'C_follicle':      C_foll_t,
    'C_dermis':        C_derm_t,
    'J_transcellular': J_transcellular*1000,
    'J_follicle':      J_follicle*1000,
})
df_out.to_csv('dermal_caffeine_pk.csv', index=False)

print('Summary:')
print('  With hair:    Cmax =', round(C_sys_hair.max()*1e3,4), 'ug/L')
print('  Without hair: Cmax =', round(C_sys_nohair.max()*1e3,4), 'ug/L')
print('  Occluded:     Cmax =', round(C_occ.max()*1e3,4), 'ug/L')
print('  Hair contribution: AUC ratio', round(AUC_hair/max(AUC_nohair,1e-8),2), 'x')
print('  F_absorbed with hair:   ', round(F_hair*100,3), '%')
print('  F_absorbed without hair:', round(F_nohair*100,3), '%')

## Key Findings

| Scenario | Cmax | F_abs | Key factor |
|---|---|---|---|
| With hair follicles | Higher | Higher | Shunt bypasses SC barrier |
| Without hair | Lower | Lower | SC is rate-limiting |
| Occluded | Highest | Highest | SC hydration + hair |
| Scalp (high density) | Highest | Most | 300 follicles/cm2 |
| Palm (hairless) | Lowest | Least | Transcellular only |

**Caffeine conclusion:** For hydrophilic compounds (low logP),
hair follicle shunting is the dominant absorption route.
SC barrier is less relevant — but follicle density matters greatly.

## MoBi Parallel Steps
1. Create spatial containers: Surface, SC, VE, Follicle, Dermis
2. Add neighborhoods: Surface↔SC, SC↔VE, VE↔Dermis, Surface↔Follicle, Follicle↔Dermis
3. Set PS values for each neighborhood (SC is smallest)
4. Set partition coefficients (SC lipophilic for hydrophobic drugs)
5. Connect Dermis to systemic PBPK (blood capillary flow)
6. Simulate: observe follicle shunt dominates for caffeine
7. Set PS_follicle=0: observe dramatic absorption decrease
8. Test body sites: vary follicle density parameter

## References
1. OSP MoBi Course: Dermal Absorption of Caffeine (v11)
2. Otberg N et al. Hair follicle route for drug delivery. J Invest Dermatol 2004
3. Lauer AC et al. Transfollicular drug delivery. Pharm Res 1995
4. Kasting GB et al. Skin absorption of caffeine. Skin Pharmacol Physiol 1992

---
*Nadia Tasnim Ahmed, PhD · github.com/ahmedn12*